# 🚗 Proyecto Final de Máster en IA: Aprendizaje por Refuerzo y Conducción Autónoma

Bienvenidos al reto final de Aprendizaje por Refuerzo (Deep RL). Vuestro objetivo es entrenar un agente capaz de conducir un vehículo autónomo en el simulador 3D **Duckietown**, utilizando únicamente datos de píxeles en bruto (cámara del salpicadero).

Un agente que memoriza un circuito es inútil. Deberéis entrenar en múltiples mapas y vuestra nota final dependerá del rendimiento del agente en un **mapa oculto** con obstáculos estáticos.

### 📋 Fases del Proyecto

* **Fase 1: Calentamiento.** Implementar Q-Learning tabular desde cero en `FrozenLake-v1`.
* **Fase 2: Baselines (DQN y PPO).** Implementar dos baselines usando la librería Stable Baselines3. Dado que Duckietown usa acciones continuas, tendréis que crear un Wrapper para discretizarlas en DQN.
* **Fase 3: Algoritmo Avanzado.** Diseñar y ajustar un tercer algoritmo de libre elección (ej. SAC, TD3, A2C, o una versión fuertemente optimizada de PPO con Curriculum Learning) que supere a los baselines.

### ⚠️ EL CONTRATO DE EVALUACIÓN (LEER ATENTAMENTE)
La evaluación está automatizada. Si vuestro código no funciona a la primera en la máquina del profesor, la nota de código será 0.
1.  **Observaciones:** El modelo DEBE esperar un espacio de observación con forma `(1, 64, 64)` apilado en 4 frames (Frame Stacking).
2.  **Clases personalizadas:** Vuestro archivo de evaluación debe incluir las definiciones exactas de las clases de Python de vuestra CNN y Wrappers.
3.  **Dry-Run:** Antes de entregar, debéis ejecutar todo en un Google Colab limpio para verificar que las dependencias funcionan.


## Fase 1: Calentamiento Tabular (Q-Learning)
Demostrad que entendéis la Ecuación de Bellman. Implementad Q-Learning desde cero (sin librerías de Deep RL) para resolver el entorno `FrozenLake-v1` (cuadrícula 8x8, resbaladiza).


In [ ]:
#Celda 1: Implementación de Q-Learning para el entorno FrozenLake-v1 (8x8, resbaladizo)
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

def train_q_learning(episodes=10000):
    # DONE: Guardar y devolver la recompensa acumulada por episodio -> Se guarda en la lista de rewards
    # DONE: Inicializar el entorno FrozenLake-v1 (8x8, resbaladizo)
    env = gym.make('FrozenLake-v1', map_name="8x8", is_slippery=True)

    # DONE: Crear la Q-Table (matriz de ceros) ->estados x acciones
    state_size = env.observation_space.n
    action_size = env.action_space.n
    q_table = np.zeros((state_size, action_size))

    # Hiperparámetros
    alpha = 0.5          # Tasa de aprendizaje
    gamma = 0.99         # Factor de descuento
    epsilon = 1.0        # Ratio de exploración inicial
    max_epsilon = 1.0
    min_epsilon = 0.01
    decay_rate = 0.0001  # Ritmo de decaimiento de epsilon

    rewards = []

    # DONE: Implementar el bucle de entrenamiento con la Ecuación de Bellman
    for episode in range(episodes):
        state, info = env.reset()
        done = False
        total_rewards = 0

        while not done:
            # Estrategia Epsilon-Greedy (Exploración vs Explotación)
            exp_exp_tradeoff = np.random.uniform(0, 1)

            if exp_exp_tradeoff > epsilon:
                # Explotación: elegir la mejor acción según la Q-table
                action = np.argmax(q_table[state, :])
            else:
                # Exploración: acción aleatoria
                action = env.action_space.sample()

            # Ejecutar acción
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

            # Ecuación de Bellman para Q-Learning
            q_table[state, action] = q_table[state, action] + alpha * (
                reward + gamma * np.max(q_table[next_state, :]) - q_table[state, action]
            )

            total_rewards += reward
            state = next_state

        # DONE: Implementar decaimiento de epsilon (Epsilon-Greedy) (reducir exploración)
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
        rewards.append(total_rewards)

    env.close()
    print("¡Entrenamiento completado!")
    return rewards

# Ejecución y visualización
rewards = train_q_learning(episodes=300000) # -> Aumentar el número de episodios para ver una curva más clara y poder sacar las conclusiones
plt.figure(figsize=(10,5))
plt.plot(np.convolve(rewards, np.ones(100)/100, mode='valid')) # Media móvil de 100 episodios
plt.title('Progreso Q-Learning en FrozenLake 8x8')
plt.xlabel('Episodios')
plt.ylabel('Recompensa Media (Ventana 100)')
plt.grid(True)
plt.show()


# Ejecución y visualización
# rewards = train_q_learning()
# plt.plot(np.convolve(rewards, np.ones(100)/100, mode='valid')) # Media móvil
# plt.title('Progreso Q-Learning')
# plt.show()

Para demostrar la correcta implementación de la ecuación de Bellman y el comportamiento asintótico del algoritmo, se realizó un experimento extendido a 300.000 episodios. Los resultados (ver gráfica) demuestran un umbral de aprendizaje crítico en torno al episodio 15.000, momento en el cual el retorno positivo de la meta se propaga con éxito hasta el estado inicial. El algoritmo converge de forma estable en una tasa de éxito de ~70%, el máximo teórico alcanzable dado el techo estocástico que impone la física resbaladiza (is_slippery=True) del escenario 8x8

### Nota de diseño: Q-Learning vs Monte Carlo
El enunciado pide **Q-Learning** (un método de diferencia temporal, TD). A modo de justificación de esa elección frente a un método de Monte Carlo, esta celda compara ambos en el mismo entorno. Monte Carlo solo aprende de episodios completos que alcanzan la meta; en un entorno disperso y muy estocástico (8x8 resbaladizo) esos episodios son rarísimos al principio, por lo que su aprendizaje es órdenes de magnitud más lento. Q-Learning, al hacer *bootstrapping* paso a paso, propaga la señal mucho antes.

In [ ]:
# (Opcional) Monte Carlo control (constant-alpha) para comparar con Q-Learning
def train_monte_carlo(episodes=80000, alpha=0.05, gamma=0.99):
    env = gym.make('FrozenLake-v1', map_name="8x8", is_slippery=True)
    nS, nA = env.observation_space.n, env.action_space.n
    q = np.zeros((nS, nA))
    eps, min_eps, max_eps, decay = 1.0, 0.01, 1.0, 0.0001
    rewards = []
    for ep in range(episodes):
        s, _ = env.reset()
        traj, done, tot = [], False, 0
        while not done:
            a = np.argmax(q[s]) if np.random.rand() > eps else env.action_space.sample()
            ns, r, term, trunc, _ = env.step(a)
            done = term or trunc
            traj.append((s, a, r)); tot += r; s = ns
        G = 0.0
        for (s, a, r) in reversed(traj):           # retorno REAL, sin bootstrapping
            G = r + gamma * G
            q[s, a] += alpha * (G - q[s, a])
        eps = min_eps + (max_eps - min_eps) * np.exp(-decay * ep)
        rewards.append(tot)
    env.close()
    return rewards

# Descomenta para ejecutar la comparación (Q-Learning ya está entrenado arriba):
rewards_mc = train_monte_carlo(300000)
w = 1000
plt.figure(figsize=(9,5))
plt.plot(np.convolve(rewards,    np.ones(w)/w, mode='valid'), label="Q-Learning (TD)")
plt.plot(np.convolve(rewards_mc, np.ones(w)/w, mode='valid'), label="Monte Carlo")
plt.xlabel("Episodios"); plt.ylabel("Tasa de éxito (media móvil)"); plt.legend(); plt.grid(alpha=0.3)
plt.title("Q-Learning vs Monte Carlo en FrozenLake 8x8 resbaladizo"); plt.show()


## Fases 2 y 3: Conducción Autónoma en Duckietown

En esta sección, construiréis el pipeline para DQN, PPO y vuestro **Algoritmo Avanzado (Fase 3)**.
Aquí tenéis el código base para inicializar la pantalla virtual que requiere Google Colab y el esqueleto de procesamiento de imágenes.

**Mapas de Entrenamiento Permitidos:**
`Duckietown-loop_empty-v0`, `Duckietown-udem1-v0`, `Duckietown-zigzag_dists-v0`, `Duckietown-small_loop-v0`, `Duckietown-straight_road-v0`.

In [ ]:
#Celda 2: Implementación de PPO para el entorno Duckietown
import os
import pyvirtualdisplay
import gym as old_gym
import gymnasium as gym
from gymnasium import spaces
import gym_duckietown
import gym_duckietown.simulator as sim_module
import cv2
import numpy as np
import torch
import torch.nn as nn
from stable_baselines3 import PPO, DQN
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
import sys
from gym_duckietown.envs import DuckietownEnv
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

# Guardia para evitar aplicar el patch más de una vez
if not getattr(sim_module.Simulator._render_img, '_patched', False):
    _original_render_img = sim_module.Simulator._render_img

    def _patched_render_img(self, width, height, multi_fbo, final_fbo, img_array, top_down=True, segment=False):
        self.cur_pos = np.array(self.cur_pos, dtype=np.float64)
        self.cur_angle = float(self.cur_angle)
        return _original_render_img(self, width, height, multi_fbo, final_fbo, img_array, top_down, segment)

    _patched_render_img._patched = True
    sim_module.Simulator._render_img = _patched_render_img
    print("Patch aplicado correctamente")
else:
    print("Patch ya estaba aplicado, omitiendo.")

In [ ]:
#Celda 3: Definición del entorno personalizado, la CNN personalizada y el entrenamiento de PPO

# TODO: Importar vuestro tercer algoritmo (ej. SAC, TD3, etc.)

# INICIAR PANTALLA VIRTUAL (Obligatorio en Colab)
display = pyvirtualdisplay.Display(visible=False, size=(800, 600))
display.start()

class DuckieWrapper(gym.Env):
    def __init__(self, env):  # Recibe la instancia directamente
        super().__init__()
        self.env = env
        self.action_space = spaces.Box(
            low=np.array([-1.0, -1.0]),
            high=np.array([1.0, 1.0]),
            dtype=np.float32
        )
        self.observation_space = spaces.Box(low=0, high=255, shape=(1, 64, 64), dtype=np.uint8)

    def reset(self, seed=42, options=None):
        obs = self.env.reset()
        if isinstance(obs, tuple):
            obs = obs[0]
        return self._process_obs(obs), {}

    def step(self, action):
        # Aseguramos que la acción es un array 1D de float64
        action = np.array(action, dtype=np.float64).flatten()
        obs, reward, done, info = self.env.step(action)
        return self._process_obs(obs), reward, done, False, info

    def _process_obs(self, obs):
        obs = obs[obs.shape[0]//2:, :, :]
        gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        resized = cv2.resize(gray, (64, 64), interpolation=cv2.INTER_AREA)
        return np.expand_dims(resized, axis=0)

    def render(self): 
        frame = self.env.render(mode='rgb_array')
        if frame is None:
            return None
        return cv2.flip(frame, 0)
     
    def close(self): self.env.close()

class CustomCNN(BaseFeaturesExtractor):
    """CNN Personalizada para extraer características de los píxeles.
    Espera una entrada (batch_size, n_stack, height, width), donde n_stack es 4 (Frame Stacking).
    """
    def __init__(self, observation_space: gym.spaces.Box, features_dim: int = 256):
        super().__init__(observation_space, features_dim)
        # DONE: Diseñar vuestra propia red convolucional aquí
        # El espacio de observación esperado después del VecFrameStack será (n_stack, 64, 64), es decir (4, 64, 64)
        n_input_channels = observation_space.shape[0]

        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 32, kernel_size=8, stride=4), # Salida: (32, 15, 15)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2), # Salida: (64, 7, 7)
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), # Salida: (64, 5, 5)
            nn.ReLU(),
            nn.Flatten(),
        )

        # Calcular el tamaño de la salida de la CNN para la capa lineal
        with torch.no_grad():
            n_flatten = self.cnn(torch.as_tensor(observation_space.sample()[None]).float()).shape[1]

        self.linear = nn.Sequential(nn.Linear(n_flatten, features_dim), nn.ReLU())

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        # DONE: Implementar el Forward pass
        # Normalizar las observaciones a un rango [0, 1] si no lo están ya (imágenes)
        return self.linear(self.cnn(observations / 255.0))

### Bucle de Entrenamiento
Entrenad vuestros modelos (DQN, PPO y el Algoritmo Avanzado). Recordad usar **Frame Stacking** (apilamiento de frames) para que el agente tenga percepción de la velocidad y giros. Guardad vuestro modelo final usando `model.save()`.

In [ ]:
#Celda 4: Entrenamiento de PPO con la CNN personalizada

import geometry.poses as poses_module
import inspect

path = inspect.getfile(poses_module)

with open(path, 'r') as f:
    content = f.read()

fixes = {
    "    t = np.array(t)\n    return combine_pieces(rot2d(theta), t, t * 0, 1)":
        "    t = np.array([float(v) for v in np.array(t).flatten()[:2]], dtype='float64')\n    return combine_pieces(rot2d(theta), t, t * 0, 1)",
    "    linear = np.array(linear, dtype='float64')":
        "    linear = np.array([float(v) for v in np.array(linear).flatten()[:2]], dtype='float64')",
}

for old, new in fixes.items():
    if old in content:
        content = content.replace(old, new)
        print(f"Fix aplicado: {old[:50]}...")
    elif new in content:
        print(f"Fix ya estaba: {old[:50]}...")
    else:
        print(f"No encontrado: {old[:50]}...")

with open(path, 'w') as f:
    f.write(content)

In [ ]:
#Celda 5: Verificación del fix aplicado

import importlib
import geometry.poses
importlib.reload(geometry.poses)
print("Módulo geometry.poses recargado")

# Verificar que el fix está activo
import inspect
src = inspect.getsource(geometry.poses.se2_from_linear_angular)
print(src)

In [ ]:
#Celda 6: Setup de entornos, policy_kwargs y callbacks

from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3 import PPO, DQN, SAC
from stable_baselines3.common.callbacks import CheckpointCallback

map_names = [
    "loop_empty",
    "udem1",
    "zigzag_dists",
    "small_loop",
    "straight_road"
]

class RewardWrapper(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        self.last_pos = None

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.last_pos = self.env.env.cur_pos.copy()
        return obs, info

    def step(self, action):
        obs, reward, done, truncated, info = self.env.step(action)
        cur_pos = self.env.env.cur_pos.copy()

        if self.last_pos is not None:
            dist = np.linalg.norm(cur_pos - self.last_pos)
            forward = info.get('forward_reward', 0)
            lane_pos = info.get('lane_position', None)

            # Premio por avanzar hacia adelante
            if forward > 0.01 and dist > 0.01:
                reward += 2.0

            # Premio por girar correctamente manteniéndose en carril
            if lane_pos is not None:
                dist_center = abs(lane_pos.get('dist', 0))
                angle_err = abs(lane_pos.get('angle_deg', 0))

                # Cuanto más centrado y alineado, mejor
                reward += max(0, 1.0 - dist_center * 2)   # hasta +1 por estar centrado
                reward += max(0, 1.0 - angle_err / 45)    # hasta +1 por estar alineado

            # Penalización por girar en círculos sin avanzar
            if dist > 0.01 and forward < 0.001:
                reward -= 3.0

            # Penalización leve por quieto
            if dist < 0.01:
                reward -= 0.5

        self.last_pos = cur_pos

        # Penalización muy fuerte por salirse
        if done:
            reward -= 200.0

        return obs, reward, done, truncated, info


def make_env(map_name):
    def _init():
        raw_env = DuckietownEnv(map_name=map_name, seed=42)
        env = DuckieWrapper(raw_env)
        env = RewardWrapper(env)
        return env
    return _init

# Entorno continuo (para PPO y SAC)
env = DummyVecEnv([make_env(name) for name in map_names])
env = VecFrameStack(env, n_stack=4)

policy_kwargs = dict(
    features_extractor_class=CustomCNN,
    features_extractor_kwargs=dict(features_dim=256),
)

checkpoint_callback_dqn = CheckpointCallback(save_freq=10000, save_path='./logs/dqn/', name_prefix='dqn_model')
checkpoint_callback_ppo = CheckpointCallback(save_freq=10000, save_path='./logs/ppo/', name_prefix='ppo_model')
checkpoint_callback_sac = CheckpointCallback(save_freq=10000, save_path='./logs/sac/', name_prefix='sac_model')

print("Setup completado.")

In [ ]:
# Celda 7: Discretización del espacio de acción para DQN (opcional, si quieres usar DQN con DuckieWrapper)

class DiscretizationWrapper(gym.Env):
    """
    Convierte el action_space continuo de DuckieWrapper en discreto.
    DQN elige un índice entero → este wrapper lo traduce a [vel, giro].
    """

    # Tabla de acciones: [velocidad, giro]
    ACTIONS = np.array([
        [ 0.7,  0.0],   # 0: adelante rápido
        [ 0.3,  0.0],   # 1: adelante lento
        [ 0.5,  0.3],   # 2: giro izquierda suave
        [ 0.5, -0.3],   # 3: giro derecha suave
        [ 0.3,  0.6],   # 4: giro izquierda fuerte
        [ 0.3, -0.6],   # 5: giro derecha fuerte
        [ 0.0,  0.0],   # 6: parar
    ], dtype=np.float32)

    def __init__(self, env: DuckieWrapper):
        super().__init__()
        self.env = env
        self.action_space = spaces.Discrete(len(self.ACTIONS))
        self.observation_space = env.observation_space  # hereda (1, 64, 64)

    def reset(self, seed=42, options=None):
        return self.env.reset(seed=seed, options=options)

    def step(self, action: int):
        continuous_action = self.ACTIONS[action]
        return self.env.step(continuous_action)

    def render(self): 
        return self.env.render()
    
    def close(self): return self.env.close()

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

In [ ]:
# Celda: Cargar modelos ya entrenados (ejecutar en lugar de las celdas 8, 9 y 10)
import os
if os.path.exists("dqn_duckie_agent.zip"):
    model_dqn = DQN.load("dqn_duckie_agent")
    print("DQN cargado.")
if os.path.exists("ppo_duckie_agent.zip"):
    model_ppo = PPO.load("ppo_duckie_agent")
    print("PPO cargado.")
if os.path.exists("sac_duckie_agent.zip"):
    model_sac = SAC.load("sac_duckie_agent")
    print("SAC cargado.")

In [ ]:
# Celda 8: Entrenamiento de DQN con la CNN personalizada y comparación con PPO y SAC

def make_env_dqn(map_name):
    def _init():
        raw_env = DuckietownEnv(map_name=map_name, seed=42)
        env = DuckieWrapper(raw_env)
        env = RewardWrapper(env)          # ← añadir esto
        env = DiscretizationWrapper(env)
        return env
    return _init

# Entorno para DQN (discreto, sin SAC/PPO)
env_dqn = DummyVecEnv([make_env_dqn(name) for name in map_names])
env_dqn = VecFrameStack(env_dqn, n_stack=4)

# DQN con tu CustomCNN
model_dqn = DQN(
    "CnnPolicy",
    env_dqn,
    verbose=0,
    device=device,
    policy_kwargs=policy_kwargs,           # reutilizas el mismo policy_kwargs
    learning_rate=1e-4,
    buffer_size=50_000,
    learning_starts=5_000,
    batch_size=32,
    exploration_fraction=0.15,
    exploration_final_eps=0.05,
    tensorboard_log="./dqn_duckietown_tensorboard/"
)

model_dqn.learn(total_timesteps=1_000_000, callback=checkpoint_callback_dqn)
model_dqn.save("dqn_duckie_agent")
print("DQN entrenado y guardado.")

In [ ]:
# Celda 9: Entrenamiento PPO
print("Entrenando PPO...")
model_ppo = PPO(
    "CnnPolicy", env,
    verbose=0,
    device=device,
    policy_kwargs=policy_kwargs,
    learning_rate=3e-4,
    n_steps=512,          
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    ent_coef=0.01,        
    tensorboard_log="./ppo_duckietown_tensorboard/"
)
model_ppo.learn(total_timesteps=1_000_000, callback=checkpoint_callback_ppo)
model_ppo.save("ppo_duckie_agent")
print("PPO entrenado y guardado.")

In [ ]:
# Celda 10: Entrenamiento SAC (Algoritmo Avanzado - Fase 3)
print("Entrenando SAC...")
model_sac = SAC(
    "CnnPolicy", env,
    verbose=0,
    device=device,
    policy_kwargs=policy_kwargs,
    learning_rate=3e-4,
    buffer_size=100_000,      # más memoria de experiencias
    learning_starts=10_000,   # espera más antes de entrenar
    batch_size=256,           # SAC funciona mejor con batches grandes
    tau=0.005,                # actualización suave del target network
    gamma=0.99,
    train_freq=1,             # actualiza en cada paso
    gradient_steps=1,
    ent_coef='auto',          # ajusta automáticamente la entropía (clave en SAC)
    target_entropy='auto',
    tensorboard_log="./sac_duckietown_tensorboard/"
)
model_sac.learn(total_timesteps=1_000_000, callback=checkpoint_callback_sac)
model_sac.save("sac_duckie_agent")
print("SAC entrenado y guardado.")

## Evaluación Visual y Generación de Video
Para que podáis ver a vuestro agente conducir, ejecutad esta celda. Creará un entorno de prueba, dejará que el agente conduzca y generará un vídeo MP4. **El profesor usará un código idéntico a este en el mapa oculto para puntuaros.**

#### Evaluación DQN 

In [ ]:
# Celda 11: Evaluación DQN
import imageio
from IPython.display import Video

def make_test_env_dqn():
    raw_env = DuckietownEnv(map_name="udem1", seed=42)
    env = DuckieWrapper(raw_env)
    return DiscretizationWrapper(env)

test_env_dqn = DummyVecEnv([make_test_env_dqn])
test_env_dqn = VecFrameStack(test_env_dqn, n_stack=4)

model_dqn = DQN.load("dqn_duckie_agent")
obs = test_env_dqn.reset()
frames_dqn = []
reward_dqn = 0

for i in range(20000):  # máximo 10 minutos a 30 FPS
    action, _ = model_dqn.predict(obs, deterministic=True)
    obs, reward, done, info = test_env_dqn.step(action)
    frame = test_env_dqn.envs[0].env.render()
    if frame is not None:
        frames_dqn.append(cv2.flip(frame, 0))
    reward_dqn += reward[0]
    if done[0]: break

test_env_dqn.close()
imageio.mimsave('eval_dqn.mp4', frames_dqn, fps=30)
print(f"DQN - Recompensa total: {reward_dqn:.2f}")
Video('eval_dqn.mp4', embed=True)

#### Evaluación PPO

In [ ]:
# Celda 12: Evaluación PPO
def make_test_env_ppo():
    raw_env = DuckietownEnv(map_name="udem1", seed=1)
    return DuckieWrapper(raw_env)

test_env_ppo = DummyVecEnv([make_test_env_ppo])
test_env_ppo = VecFrameStack(test_env_ppo, n_stack=4)

obs = test_env_ppo.reset()
frames_ppo = []
reward_ppo = 0

for i in range(20000):  # máximo 10 minutos a 30 FPS
    action, _ = model_ppo.predict(obs, deterministic=True)
    obs, reward, done, info = test_env_ppo.step(action)
    frame = test_env_ppo.envs[0].env.render()
    if frame is not None:
        frames_ppo.append(cv2.flip(frame, 0))
    reward_ppo += reward[0]
    if done[0]: break

test_env_ppo.close()
imageio.mimsave('eval_ppo.mp4', frames_ppo, fps=30)
print(f"PPO - Recompensa total: {reward_ppo:.2f}")
Video('eval_ppo.mp4', embed=True)

In [ ]:
# Celda 13: Evaluación SAC
def make_test_env_sac():
    raw_env = DuckietownEnv(map_name="udem1", seed=42)
    return DuckieWrapper(raw_env)

test_env_sac = DummyVecEnv([make_test_env_sac])
test_env_sac = VecFrameStack(test_env_sac, n_stack=4)

model_sac = SAC.load("sac_duckie_agent")
obs = test_env_sac.reset()
frames_sac = []
reward_sac = 0

for i in range(20000):  # máximo 10 minutos a 30 FPS
    action, _ = model_sac.predict(obs, deterministic=True)
    obs, reward, done, info = test_env_sac.step(action)
    frame = test_env_sac.envs[0].env.render()
    if frame is not None:
        frames_sac.append(cv2.flip(frame, 0))
    reward_sac += reward[0]
    if done[0]: break

test_env_sac.close()
imageio.mimsave('eval_sac.mp4', frames_sac, fps=30)
print(f"SAC - Recompensa total: {reward_sac:.2f}")
Video('eval_sac.mp4', embed=True)

#### Comparación y guardado del mejor:

In [ ]:
# Celda 14: Comparación y guardado del mejor agente

# Comparar por pasos aguantados (más fiable que la recompensa con penalizaciones artificiales)
steps = {"DQN": len(frames_dqn), "PPO": len(frames_ppo), "SAC": len(frames_sac)}
scores = {"DQN": reward_dqn, "PPO": reward_ppo, "SAC": reward_sac}
best_name = max(steps, key=steps.get)

print("=== Resultados ===")
for name in ["DQN", "PPO", "SAC"]:
    print(f"  {name}: {steps[name]} pasos | recompensa: {scores[name]:.2f} {'← MEJOR' if name == best_name else ''}")

# Guardar el mejor como best_duckie_agent
models = {"DQN": model_dqn, "PPO": model_ppo, "SAC": model_sac}
models[best_name].save("best_duckie_agent")
print(f"\nMejor agente ({best_name}) guardado como best_duckie_agent.zip")

# Mostrar su video
videos = {"DQN": "eval_dqn.mp4", "PPO": "eval_ppo.mp4", "SAC": "eval_sac.mp4"}
Video(videos[best_name], embed=True)

---

## 📦 Requisitos de Entrega y Proceso de Evaluación

Estimados alumnos,

Para asegurar que el proceso de corrección sea justo y ágil, debéis seguir estrictamente las siguientes directrices para vuestra entrega final.

### 1. Entregables Obligatorios

Vuestra entrega debe ser un único archivo comprimido que contenga exactamente lo siguiente:

* **El Código Fuente:** Vuestro cuaderno `.ipynb` completo (o, en su defecto, los scripts de Python `train.py` y `eval.ipynb`).
* **Dependencias:** Un archivo `requirements.txt` con las versiones de las librerías fijadas **estrictamente** usando `==` (por ejemplo, `stable-baselines3==2.2.1`). Esto es vital para evitar problemas de compatibilidad.
* **El Agente Entrenado:** El archivo comprimido de vuestro modelo final (debe llamarse `best_duckie_agent.zip`).
* **La Memoria Técnica:** Un documento `Report.pdf`. Aquí debéis justificar teórica y empíricamente la elección y configuración de vuestro tercer algoritmo (Fase 3) y comparar detalladamente su rendimiento frente a las baselines obligatorias de DQN y PPO.

### 2. Cómo seréis evaluados (El Contrato de Evaluación)

Quiero ser muy transparente sobre cómo probaré vuestros proyectos: **no voy a depurar vuestro código ni a re-entrenar vuestros modelos desde cero.**

El proceso de corrección será exactamente el siguiente:

1. Tomaré vuestro archivo `best_duckie_agent.zip`.
2. Lo cargaré en mi propio script de evaluación, ejecutándolo en un entorno limpio basado estrictamente en vuestro `requirements.txt`.
3. Ejecutaré la celda de evaluación, pero cambiando el mapa al escenario oculto que no habéis visto durante el entrenamiento: **`Duckietown-loop_obstacles-v0`**.

Si habéis entrenado el modelo respetando las reglas de forma (mismas dimensiones de entrada, variables definidas y clases guardadas correctamente en el código), vuestro agente se cargará a la primera. Veré la simulación de vuestro coche intentando sortear los obstáculos y seréis puntuados por su capacidad de generalización y supervivencia en este nuevo mapa.

**⚠️ Advertencia final:** Si al intentar cargar vuestro modelo el script falla por incompatibilidad de versiones, clases no encontradas o errores de dimensiones en los tensores (*shape errors*), **la calificación de la parte práctica será un 0**.

Os recomiendo encarecidamente que abráis un Google Colab completamente en blanco, instaléis solo lo que dice vuestro `requirements.txt`, subáis vuestro `.zip` e intentéis ejecutar la evaluación vosotros mismos antes de enviar el trabajo definitivo.

## ¡Mucho ánimo con el entrenamiento y buena suerte!